# Sampling choices: DDPM, DDIM, and latent diffusion

Training learns a denoising field. Sampling chooses how to follow that field back toward clean data. This notebook makes the distinction concrete with a one-dimensional reverse process, then maps the same ideas onto latent diffusion systems.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='notebook')
rng = np.random.default_rng(10)

def score(x):
    # Score-like vector field with two preferred modes at -1 and +1.
    return -0.8 * (x - np.where(x < 0, -1.0, 1.0))

def sample(steps=40, stochastic=True, n=400):
    x = rng.normal(0, 2.2, n)
    trajectory = [x.copy()]
    for step in range(steps):
        step_size = 0.06 + 0.10 * (1 - step / steps)
        x = x + step_size * score(x)
        if stochastic:
            x = x + rng.normal(0, 0.08 * (1 - step / steps), n)
        trajectory.append(x.copy())
    return np.array(trajectory)

ddpm = sample(stochastic=True); ddim = sample(stochastic=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, trajectory, title in zip(axes, [ddpm, ddim], ['stochastic reverse path', 'deterministic reverse path']):
    ax.plot(trajectory[:, :35], alpha=.18, color='steelblue')
    ax.axhline(-1, color='black', ls=':'); ax.axhline(1, color='black', ls=':')
    ax.set(title=title, xlabel='reverse step', ylabel='sample value')
plt.tight_layout()

DDPM-style sampling injects noise at each reverse step, so repeated runs can produce different trajectories. DDIM-style sampling can use a deterministic path for the same initial noise and often reaches a useful sample in fewer steps. These are sampling procedures, not different training datasets.

## From pixels to latent diffusion

Latent diffusion adds an encoder and decoder around the diffusion model:

1. The encoder maps an image to a lower-dimensional latent representation.
2. The diffusion model predicts noise in that latent space.
3. A text encoder supplies conditioning through cross-attention.
4. The decoder maps the denoised latent back to pixels.

The compression changes the computational cost; it does not remove the denoising objective. Stable Diffusion is therefore a complete system assembled from an autoencoder, a text encoder, a U-Net, a noise schedule, and a sampler.

In [ ]:
components = pd = {
    'autoencoder': 'image ↔ latent',
    'text encoder': 'prompt → conditioning vectors',
    'U-Net': 'latent + time + conditioning → predicted noise',
    'sampler': 'repeated reverse updates',
}
for name, role in components.items(): print(f'{name:14} {role}')

A serious extension would compare schedules, classifier-free guidance scales, and Karras-style preconditioning on a fixed evaluation set. That belongs in a separate experiment note because sampler changes, conditioning changes, and model changes should not be mixed into one result.